# Model Training & Evaluation — CRISP-DM Phases 4 & 5

Trains all five model types and compares them on business-aligned metrics:
- Standard ML metrics: AUC-ROC, Average Precision, F1
- **Business metric**: estimated savings vs. naive baseline (PLN)
- Threshold optimisation via Precision-Recall curve
- SHAP feature importance

In [ ]:
import sys; sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data.generator import SyntheticDataGenerator
from src.data.preprocessor import DataPreprocessor, TARGET_COLUMN
from src.features.engineering import FeatureEngineer
from src.models.classical import LogisticRegressionModel, RandomForestModel, XGBoostModel
from src.models.deep_learning import NeuralNetworkModel
from src.evaluation.metrics import ModelEvaluator, BusinessMetrics
from src.visualization.plots import ChurnPlotter

plotter = ChurnPlotter()
evaluator = ModelEvaluator(BusinessMetrics(cost_false_negative=500, cost_false_positive=50))

## Prepare Data

In [ ]:
df = SyntheticDataGenerator(42).generate(n_samples=20_000)
df = FeatureEngineer().fit_transform(df)

y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN, 'customer_id',
                      'monthly_charge_tier', 'tenure_segment'], errors='ignore')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val   = train_test_split(X_train, y_train, test_size=0.1, stratify=y_train, random_state=42)

preprocessor = DataPreprocessor()
X_train_p, _ = preprocessor.fit_transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)

print(f'Train: {len(X_train_p):,} | Val: {len(X_val_p):,} | Test: {len(X_test_p):,}')

## Train Models

In [ ]:
models = [
    LogisticRegressionModel(),
    RandomForestModel(n_estimators=200),
    XGBoostModel(n_estimators=300, early_stopping_rounds=20),
    NeuralNetworkModel(hidden_dims=[128, 64], max_epochs=30, patience=5),
]

results = []
probas  = []
for model in models:
    print(f'Training {model.metadata.name}...')
    model.fit(X_train_p, y_train, X_val_p, y_val)
    proba = model.predict_proba(X_test_p)
    result = evaluator.evaluate(proba, y_test, model.metadata.name)
    results.append(result)
    probas.append((model.metadata.name, y_test.to_numpy(), proba))
    print(f'  AUC: {result.roc_auc:.4f} | F1: {result.f1:.4f} | savings: {result.business_savings:,.0f} PLN')

## Compare Models

In [ ]:
comparison = evaluator.compare(results)
comparison.style.background_gradient(subset=['roc_auc', 'f1', 'business_savings_pln'], cmap='Greens')

## ROC & Precision-Recall Curves

In [ ]:
plotter.roc_curves(probas).show()
plotter.precision_recall_curves(probas).show()

## Score Distribution — Best Model

In [ ]:
# XGBoost typically best
best_model = models[2]  # XGBoost
best_proba = best_model.predict_proba(X_test_p)
plotter.score_distribution(best_proba, y_test.to_numpy()).show()

## Feature Importance (XGBoost)

In [ ]:
importance = best_model.feature_importance(importance_type='gain')
plotter.feature_importance(importance, top_n=15, title='XGBoost Feature Importance (Gain)').show()